# Revision 2 — teacher and teacher-ensemble evaluation

Reviewer 1 comment 8: report individual teacher and ensemble performance on UCI-HAR and corrected WISDM.

Use all ten original seeds, twenty banks and one hundred saved teachers. **Zero training fits.** Preserve original production results and original Shapley weights. Do not use weighting-pilot choices or choose weights/temperature from test outcomes.

Primary ensembles average per-teacher probabilities at **inference temperature 1**, either uniformly or using the original per-bank standardized-Shapley weights. Secondary ensembles repeat the same aggregation at **temperature 4**, matching the KD-target softening used by the student. Softmax is applied per teacher before averaging; there is no mean-logit or majority-vote substitution. Predict with argmax of the resulting probabilities. Individual teachers use argmax of raw logits. Exact class ties use the first saved class index. All four ensembles are retained.

First verify every frozen file and original split, then reproduce all one hundred saved teacher validation accuracy/macro-F1 values within the unchanged absolute tolerance 0.0001. Cache raw validation logits and split arrays. No test inference occurs until every validation preflight passes. The separate smoke evaluates seed 11 on both datasets; it is excluded from research. Research evaluates all banks and also checks teacher test metrics against source records at the same tolerance.

Research reports five teachers and four ensembles per bank: **180 condition results / 360 validation-and-test metric sets**. Save logits, predictions, labels, subject IDs, configurations and completion hashes. These are descriptive post-review results on already-used benchmark splits. Ten seeds describe model variability conditional on the subject split. No new significance or independent-generalization claim is planned. This evaluation is not the deployment-latency benchmark.

Run cells in order on a GPU. Compatible evaluations resume after identity/hash checks. Stop on any original-source, split or metric-parity mismatch; do not relax tolerance. Models and prediction arrays stay on Google Drive; OneDrive holds later verified tables/figures and selected small evidence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use the persistent Google Drive project; no temporary upload is required.
from pathlib import Path
PROJECT=Path('/content/drive/MyDrive/makd-har')
EXPERIMENT=PROJECT/'revision_2/teacher_ensemble'
assert (EXPERIMENT/'frozen-inputs.json').is_file(), 'Google Drive is not mounted or the project has not synchronized. Run the mount cell above first.'
print('Project:',PROJECT)
print('Inputs and results use persistent Google Drive storage. Frozen-file hashes are checked in the preflight cell.')

In [ ]:
import subprocess,sys,json,os,datetime
import tensorflow as tf
print('TensorFlow',tf.__version__,'Keras',tf.keras.__version__)
print('GPU',tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):raise RuntimeError('Choose a GPU runtime before continuing.')

def run_ensemble(*flags):
    label='verify' if '--verify-only' in flags else ('preflight' if '--preflight-only' in flags else ('smoke' if '--smoke' in flags else 'research'))
    logs=PROJECT/'results/revision-2/diagnostics';logs.mkdir(parents=True,exist_ok=True)
    log=logs/f"teacher-ensemble-{label}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    print('Full diagnostic log:',log,flush=True)
    command=[sys.executable,'-u',str(EXPERIMENT/'run_ensemble.py'),'--drive-root',str(PROJECT),*flags]
    notices=0
    with log.open('w') as stream:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,
                              env=dict(os.environ,TF_CPP_MIN_LOG_LEVEL='2')) as process:
            for line in process.stdout:
                stream.write(line);stream.flush()
                if 'NodeDef mentions attribute use_unbounded_threadpool' in line:notices+=1
                else:print(line,end='',flush=True)
            code=process.wait()
    if notices:print(f'{notices} repeated TensorFlow compatibility notices retained in the full log.')
    if code:raise RuntimeError(f'{label} stopped (exit {code}). Full error saved in {log}.')


In [ ]:
# Verify source/data hashes, original split and all 100 teacher validation records; no test inference.
run_ensemble('--verify-only')
run_ensemble('--preflight-only')
gate=json.loads((PROJECT/'results/revision-2/teacher-ensemble/preflight-completed.json').read_text())
assert gate['passed'] and gate['teacher_checks']==100 and len(gate['files'])==44
print('PREFLIGHT PASSED: all one hundred saved teachers match original validation records.')


In [ ]:
# Separate inference smoke: seed 11 on both datasets, all teachers and four ensembles.
run_ensemble('--smoke')
record=json.loads((PROJECT/'results/revision-2/smoke-teacher-ensemble/completed.json').read_text())
assert record['status']=='complete' and record['mode']=='smoke' and record['banks']==2 and record['training_fits']==0
print('SMOKE PASSED: both datasets; ten teachers and eight ensembles; no training.')


In [ ]:
# Full inference evaluation: twenty banks; no teacher/student retraining.
run_ensemble()


In [ ]:
completed=PROJECT/'results/revision-2/teacher-ensemble/completed.json'
assert completed.is_file(), 'Incomplete: rerun only the research evaluation cell to resume verified banks.'
record=json.loads(completed.read_text())
assert record['status']=='complete' and record['mode']=='research' and record['banks']==20
assert record['teacher_evaluations']==100 and record['ensemble_evaluations']==80 and record['metric_sets']==360
assert record['training_fits']==0
print('COMPLETE: 100 teachers, 80 ensembles, 360 metric sets. Ready for OneDrive verification.')
